In [ ]:
import requests
import json

def print_keys(data, path=""):
    """
    Recursively finds and prints all keys in a nested dictionary/list.
    """
    if isinstance(data, dict):
        for key, value in data.items():
            # Build the path string (e.g., "user -> profile -> name")
            new_path = f"{path} -> {key}" if path else key
            print(new_path)
            # Dive deeper
            print_keys(value, new_path)
            
    elif isinstance(data, list):
        for index, item in enumerate(data):
            # We don't usually name list indices as 'keys', 
            # but we need to check if the items inside have keys.
            print_keys(item, f"{path}[{index}]")

# Usage:
# data = response.json()
# print_keys(data)

url = "https://api.elections.kalshi.com/trade-api/v2/events/KXMARMAD-26"
params = {
    'with_nested_markets': 'true',
}

response = requests.get(url, params) # get(url, params) where params = {'...':'value'}
data = response.json()
print_keys(data)
# print(json.dumps(data,indent=4))

event
event -> available_on_brokers
event -> category
event -> collateral_return_type
event -> event_ticker
event -> last_updated_ts
event -> markets
event -> markets[0] -> can_close_early
event -> markets[0] -> close_time
event -> markets[0] -> created_time
event -> markets[0] -> custom_strike
event -> markets[0] -> custom_strike -> basketball_team
event -> markets[0] -> early_close_condition
event -> markets[0] -> event_ticker
event -> markets[0] -> expected_expiration_time
event -> markets[0] -> expiration_time
event -> markets[0] -> expiration_value
event -> markets[0] -> fractional_trading_enabled
event -> markets[0] -> last_price_dollars
event -> markets[0] -> latest_expiration_time
event -> markets[0] -> liquidity_dollars
event -> markets[0] -> market_type
event -> markets[0] -> no_ask_dollars
event -> markets[0] -> no_bid_dollars
event -> markets[0] -> no_sub_title
event -> markets[0] -> notional_value_dollars
event -> markets[0] -> open_interest_fp
event -> markets[0] -> open_

In [ ]:
import asyncio
import httpx
import pandas as pd
from tqdm.asyncio import tqdm

# Config
EVENTS = ["KXMARMAD-25", "KXMARMAD-26"]
BASE_URL = "https://api.elections.kalshi.com/trade-api/v2"

async def get_team_mapping():
    mapping_data = []
    
    async with httpx.AsyncClient() as client:
        for event_ticker in EVENTS:
            print(f"Resolving teams for {event_ticker}...")
            
            # Check both Market Discovery endpoints
            for path in ["/markets", "/historical/markets"]:
                params = {"event_ticker": event_ticker, "limit": 1000}
                
                # Handling pagination for market discovery
                cursor = None
                while True:
                    if cursor: params["cursor"] = cursor
                    
                    resp = await client.get(f"{BASE_URL}{path}", params=params)
                    if resp.status_code != 200:
                        break
                    
                    data = resp.json()
                    markets = data.get('markets', [])
                    
                    for m in markets:
                        # Priority: yes_sub_title -> no_sub_title -> title
                        team_name = (
                            m.get('yes_sub_title') or 
                            m.get('no_sub_title') or 
                            m.get('title')
                        )
                        
                        mapping_data.append({
                            "ticker": m.get('ticker'),
                            "event": event_ticker,
                            "team_name": team_name,
                            "market_type": "historical" if "historical" in path else "live"
                        })
                    
                    cursor = data.get('cursor')
                    if not cursor or not markets:
                        break
                        
    return pd.DataFrame(mapping_data)

# Jupyter execution
team_mapping_df = await get_team_mapping()

# Clean up: Often the sub_title has extra whitespace or "Yes/No" artifacts
team_mapping_df['team_name'] = team_mapping_df['team_name'].str.strip()

print(f"Mapped {len(team_mapping_df)} tickers to team names.")
team_mapping_df.tail(10)
# team_mapping_df.to_csv('/Users/michaelharoon/Projects/tasty/march-madness/data/market_data_store/name_maping.csv')

Resolving teams for KXMARMAD-25...
Resolving teams for KXMARMAD-26...
Mapped 181 tickers to team names.


,ticker,event,team_name,market_type
171,KXMARMAD-26-UGA,KXMARMAD-26,Georgia,live
172,KXMARMAD-26-MIZZ,KXMARMAD-26,Missouri,live
173,KXMARMAD-26-PV,KXMARMAD-26,Prairie View A&M,live
174,KXMARMAD-26-MOH,KXMARMAD-26,Miami (OH),live
175,KXMARMAD-26-FUR,KXMARMAD-26,Furman,live
176,KXMARMAD-26-SLU,KXMARMAD-26,Saint Louis,live
177,KXMARMAD-26-MIA,KXMARMAD-26,Miami (FL),live
178,KXMARMAD-26-GONZ,KXMARMAD-26,Gonzaga,live
179,KXMARMAD-26-VAN,KXMARMAD-26,Vanderbilt,live
180,KXMARMAD-26-KU,KXMARMAD-26,Kansas,live


In [ ]:
import requests
import json

# Fetch data
hurl = "https://api.elections.kalshi.com/trade-api/v2/historical/trades?limit=500&ticker=KXMARMAD-25-DU"
hresponse = requests.get(hurl)

url = "https://api.elections.kalshi.com/trade-api/v2/markets/trades?limit=500&ticker=KXMARMAD-25-DU"
response = requests.get(url)

# 1. Extract JSON data
h_data = hresponse.json()
data = response.json()

def compare_structures(dict1, dict2, name1="Historical", name2="Live"):
    print(f"--- Structural Comparison: {name1} vs {name2} ---")
    
    # Compare Top Level Keys
    keys1 = set(dict1.keys())
    keys2 = set(dict2.keys())
    
    print(f"Common keys: {keys1 & keys2}")
    if keys1 - keys2:
        print(f"Keys only in {name1}: {keys1 - keys2}")
    if keys2 - keys1:
        print(f"Keys only in {name2}: {keys2 - keys1}")
    
    # Compare keys of the first item in 'trades' list (if exists)
    list_key = 'trades' # Kalshi usually wraps results in a 'trades' key
    if list_key in dict1 and list_key in dict2:
        if dict1[list_key] and dict2[list_key]:
            t1_keys = set(dict1[list_key][0].keys())
            t2_keys = set(dict2[list_key][0].keys())
            
            print(f"\n--- Individual Trade Object Keys ---")
            print(f"Common trade keys: {t1_keys & t2_keys}")
            if t1_keys - t2_keys:
                print(f"Trade keys only in {name1}: {t1_keys - t2_keys}")
            if t2_keys - t1_keys:
                print(f"Trade keys only in {name2}: {t2_keys - t1_keys}")
        else:
            print(f"\nOne of the 'trades' lists is empty.")

# Run comparison
compare_structures(h_data, data)

# 2. Check if the entire data payload is identical
print(f"\nAre payloads identical? {h_data == data}")

# 3. Deep Dive into the differences
h_trades = h_data.get('trades', [])
l_trades = data.get('trades', [])

# Convert trade lists to IDs for comparison
h_ids = {t['trade_id'] for t in h_trades}
l_ids = {t['trade_id'] for t in l_trades}

print(f"--- Data Content Deep Dive ---")
print(f"Total Historical trades: {len(h_trades)}")
print(f"Total Live trades: {len(l_trades)}")

# Check for ID overlap
common_ids = h_ids & l_ids
print(f"Number of identical Trade IDs in both: {len(common_ids)}")

if len(common_ids) > 0:
    # Pick one common trade and see if the values match
    sample_id = list(common_ids)[0]
    h_sample = next(t for t in h_trades if t['trade_id'] == sample_id)
    l_sample = next(t for t in l_trades if t['trade_id'] == sample_id)
    
    if h_sample == l_sample:
        print(f"Result: The trades with the same ID have identical data.")
    else:
        print(f"Result: Found a trade (ID: {sample_id}) where values differ!")
        # Print the specific field differences
        for key in h_sample:
            if h_sample[key] != l_sample[key]:
                print(f"  Field '{key}': Historical={h_sample[key]}, Live={l_sample[key]}")

# Check for unique trades
print(f"Trades only in Historical: {len(h_ids - l_ids)}")
print(f"Trades only in Live: {len(l_ids - h_ids)}")

--- Structural Comparison: Historical vs Live ---
Common keys: {'cursor', 'trades'}

One of the 'trades' lists is empty.

Are payloads identical? False
--- Data Content Deep Dive ---
Total Historical trades: 500
Total Live trades: 0
Number of identical Trade IDs in both: 0
Trades only in Historical: 500
Trades only in Live: 0


In [ ]:
import asyncio
import httpx
import pandas as pd
from tqdm.asyncio import tqdm

# Config
EVENTS = ["KXMARMAD-25", "KXMARMAD-26"]
BASE_URL = "https://api.elections.kalshi.com/trade-api/v2"
CONCURRENCY_LIMIT = 5 # Lower is safer for laptop RAM and Rate Limits

async def fetch_with_backoff(client, url, params, ticker_label=""):
    """Fetch a single page with retries on rate limits (429)."""
    for attempt in range(5): # Up to 5 retries
        try:
            resp = await client.get(url, params=params, timeout=30.0)
            if resp.status_code == 200:
                return resp.json()
            elif resp.status_code == 429:
                wait = (2 ** attempt) + 1
                # print(f"  [Rate Limit] {ticker_label} waiting {wait}s...")
                await asyncio.sleep(wait)
            else:
                print(f"  [Error {resp.status_code}] {ticker_label}")
                return None
        except Exception as e:
            await asyncio.sleep(1)
    return None

async def fetch_full_history(client, ticker, endpoint_url, semaphore):
    """Exhaustively paginate through every trade for a ticker."""
    all_trades = []
    cursor = None
    
    async with semaphore:
        while True:
            params = {"ticker": ticker, "limit": 1000}
            if cursor: params["cursor"] = cursor
            
            data = await fetch_with_backoff(client, endpoint_url, params, ticker)
            if not data: break
            
            batch = data.get('trades', [])
            for t in batch: t['ticker'] = ticker # Metadata injection
            all_trades.extend(batch)
            
            cursor = data.get('cursor')
            # Kalshi returns empty string or None when finished
            if not cursor or not batch:
                break
                
    return all_trades

async def run_exhaustive_extraction():
    # Use a limits object to prevent connection pool exhaustion
    limits = httpx.Limits(max_keepalive_connections=5, max_connections=10)
    async with httpx.AsyncClient(limits=limits) as client:
        semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)
        
        # 1. DISCOVERY: Find every ticker via paginated market calls
        all_tickers = set()
        for event in EVENTS:
            for m_path in ["/markets", "/historical/markets"]:
                m_cursor = None
                while True:
                    m_params = {"event_ticker": event, "limit": 1000}
                    if m_cursor: m_params["cursor"] = m_cursor
                    
                    m_data = await fetch_with_backoff(client, f"{BASE_URL}{m_path}", m_params, f"Discovery-{event}")
                    if not m_data: break
                    
                    batch = m_data.get('markets', [])
                    all_tickers.update([m['ticker'] for m in batch])
                    m_cursor = m_data.get('cursor')
                    if not m_cursor or not batch: break

        print(f"Discovered {len(all_tickers)} unique tickers. Starting Trade Extraction...")

        # 2. EXTRACTION: Create tasks for BOTH endpoints for EVERY ticker
        # We use a dict to keep track of which ticker belongs to which result
        h_tasks = {ticker: fetch_full_history(client, ticker, f"{BASE_URL}/historical/trades", semaphore) for ticker in all_tickers}
        m_tasks = {ticker: fetch_full_history(client, ticker, f"{BASE_URL}/markets/trades", semaphore) for ticker in all_tickers}

        # Gather results
        print("Fetching from Historical Endpoint...")
        h_results = await tqdm.gather(*h_tasks.values(), desc="Historical API")
        
        print("Fetching from Markets Endpoint...")
        m_results = await tqdm.gather(*m_tasks.values(), desc="Markets API")

        # Flatten
        historical_final = [trade for sublist in h_results for trade in sublist]
        markets_final = [trade for sublist in m_results for trade in sublist]
        
        return historical_final, markets_final

historical_trades, markets_trades = await run_exhaustive_extraction()

Discovered 181 unique tickers. Starting Trade Extraction...
Fetching from Historical Endpoint...


Historical API: 100%|██████████| 181/181 [00:22<00:00,  8.19it/s]


Fetching from Markets Endpoint...


Markets API: 100%|██████████| 181/181 [00:34<00:00,  5.26it/s]


In [ ]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import os

def save_to_endpoint_store(data, endpoint_name):
    """Writes a list of trade dicts to its specific sub-directory."""
    if not data:
        return
    
    base_path = f"data/market_data_store/{endpoint_name}"
    os.makedirs(base_path, exist_ok=True)
    
    df = pd.DataFrame(data)
    # Microstructure essentials: cast to high-precision types
    df['created_time'] = pd.to_datetime(df['created_time'])
    df['year'] = df['created_time'].dt.year
    
    # Write partitioned by year and ticker for fast lookups
    table = pa.Table.from_pandas(df)
    pq.write_to_dataset(
        table,
        root_path=base_path,
        partition_cols=['year', 'ticker'],
        compression='snappy'
    )
    print(f"Stored {len(df)} trades in {endpoint_name}")

# Example execution within your async loop
save_to_endpoint_store(historical_trades, "historical-endpoint")
save_to_endpoint_store(markets_trades, "markets-endpoint")

Stored 251221 trades in historical-endpoint
Stored 444946 trades in markets-endpoint
